# Pick 1 vs picks 7 and 12

If Port Adelaide's academy prospect Dougie Cochrane attracts a bid at pick 1,
matching it costs about **picks 7 and 12** (2,700 points after the 10%
discount for finishing 11th–18th, against 1,543 + 1,140 = 2,683). This notebook
asks whether pick 1 has historically been worth that.

**Section 1** looks at every pick 1 in the data: how many went on to have a star
season, and how many had more than one.

Star seasons use the definitions already fixed in
[star_curve_definitions.md](../docs/star_curve_definitions.md): the
All-Australian squad (headline), the Brownlow top 36, the Brownlow top 40, and
squad or Brownlow top 40. The population is national pick 1s from the 2005–2018 drafts, so
every player has a complete 8-season window. Career counts (every season to 2026)
are shown alongside for context.

In [1]:
import pandas as pd

from afl.afltables import all_season_games
from afl.datasets import draft_outcomes
from afl.stars import all_australian_squad, star_season_counts

DEFINITIONS = {"squad": "All-Australian squad", "brownlow36": "Brownlow top 36",
               "brownlow40": "Brownlow top 40", "either40": "Squad or Brownlow top 40"}
pd.set_option("display.width", 220)

seasons = all_season_games(2006, 2026)
squad = all_australian_squad(2006, 2026)
picks = draft_outcomes(2005, 2018, windows=(8,))
print(f"National picks 1-80, 2005-2018: {len(picks)}")

National picks 1-80, 2005-2018: 1004


## 1. Every pick 1, 2005–2018

In [2]:
pick1 = picks[picks["pick"] == 1].copy()
within_8 = star_season_counts(pick1, seasons, squad, years=8)
career = star_season_counts(pick1, seasons, squad, years=None)

table = pick1[["draft_year", "player", "club", "height_cm", "games_8"]].join(
    within_8.add_suffix(" (8)")).join(career[["squad", "either40"]].add_suffix(" (career)"))
table.sort_values("draft_year").rename(columns={"games_8": "games (8)"}).set_index("draft_year")

,player,club,height_cm,games (8),squad (8),brownlow36 (8),brownlow40 (8),either40 (8),squad (career),either40 (career)
draft_year,,,,,,,,,,
2005,Marc Murphy,Carlton,180.0,165.0,2,5,5,5,2,7
2006,Bryce Gibbs,Carlton,188.0,177.0,1,2,3,3,1,5
2007,Matthew Kreuzer,Carlton,200.0,119.0,0,0,0,0,1,1
2008,Jack Watts,Melbourne,196.0,137.0,0,0,0,0,0,0
2009,Tom Scully,Melbourne,181.0,151.0,2,0,0,2,2,2
2010,David Swallow,Gold Coast,186.0,117.0,0,0,0,0,0,0
2011,Jonathon Patton,GWS,197.0,89.0,0,0,0,0,0,0
2012,Lachie Whitfield,GWS,187.0,150.0,1,2,2,2,2,3
2013,Tom Boyd,GWS,201.0,61.0,0,0,0,0,0,0


In [3]:
rows = []
for col, label in DEFINITIONS.items():
    rows.append({
        "definition": label,
        "1+ star seasons (8)": f"{(within_8[col] >= 1).sum()} of {len(pick1)}",
        "2+ star seasons (8)": f"{(within_8[col] >= 2).sum()} of {len(pick1)}",
        "1+ (career)": f"{(career[col] >= 1).sum()} of {len(pick1)}",
        "2+ (career)": f"{(career[col] >= 2).sum()} of {len(pick1)}",
    })
pd.DataFrame(rows).set_index("definition")

,1+ star seasons (8),2+ star seasons (8),1+ (career),2+ (career)
definition,,,,
All-Australian squad,7 of 14,4 of 14,8 of 14,5 of 14
Brownlow top 36,4 of 14,4 of 14,4 of 14,4 of 14
Brownlow top 40,5 of 14,4 of 14,5 of 14,4 of 14
Squad or Brownlow top 40,8 of 14,6 of 14,9 of 14,6 of 14


## 2. Picks 5–15: chance of a star season

The share of picks with at least one season in the All-Australian squad or the
Brownlow top 40 (the "either" definition) within 8 seasons. Each pick number has
only 14 players, so the **±5 curve** from the definitions (all picks within 5 of
that number) is shown next to the single-pick rate, with a 95% bootstrap
interval (2,000 resamples of players, seed 2026).

In [4]:
import numpy as np

STAR_WINDOW = 5
N_BOOT = 2000

picks["either40_star"] = (star_season_counts(picks, seasons, squad, years=8)["either40"] >= 1).astype(float)


def curve(pick, outcome):
    """Share of picks within +/- STAR_WINDOW of each pick number (1-80) with outcome == 1."""
    n = np.bincount(pick, minlength=81)[1:]
    hits = np.bincount(pick, weights=outcome, minlength=81)[1:]
    kernel = np.ones(2 * STAR_WINDOW + 1)
    return np.convolve(hits, kernel, "same") / np.convolve(n, kernel, "same")


pick, outcome = picks["pick"].to_numpy(), picks["either40_star"].to_numpy()
smoothed = curve(pick, outcome)
rng = np.random.default_rng(2026)
boots = []
for _ in range(N_BOOT):
    i = rng.integers(0, len(pick), len(pick))
    boots.append(curve(pick[i], outcome[i]))
low, high = np.percentile(boots, [2.5, 97.5], axis=0)

rows = []
for p in [1] + list(range(5, 16)):
    g = picks[picks["pick"] == p]
    rows.append({"pick": p, "n": len(g),
                 "stars": f"{int(g['either40_star'].sum())} of {len(g)}",
                 "this pick %": round(g["either40_star"].mean() * 100),
                 "±5 curve %": round(smoothed[p - 1] * 100),
                 "±5 CI": f"{low[p - 1] * 100:.0f}–{high[p - 1] * 100:.0f}"})
pd.DataFrame(rows).set_index("pick")

,n,stars,this pick %,±5 curve %,±5 CI
pick,,,,,
1,14,8 of 14,57,49,38–60
5,14,8 of 14,57,44,35–52
6,14,3 of 14,21,43,35–51
7,14,6 of 14,43,42,34–51
8,14,2 of 14,14,42,34–50
9,14,8 of 14,57,40,32–48
10,14,4 of 14,29,39,32–47
11,14,5 of 14,36,36,28–44
12,14,7 of 14,50,35,28–43


In [5]:
p1, p7, p12 = smoothed[0], smoothed[6], smoothed[11]
print(f"±5 curve: pick 1 {p1:.0%}  |  pick 7 {p7:.0%}  |  pick 12 {p12:.0%}")
print(f"At least one star from picks 7 and 12 (if independent): {1 - (1 - p7) * (1 - p12):.0%}")
print(f"Expected number of players with a star season from 7 and 12: {p7 + p12:.2f}")

±5 curve: pick 1 49%  |  pick 7 42%  |  pick 12 35%
At least one star from picks 7 and 12 (if independent): 62%
Expected number of players with a star season from 7 and 12: 0.77
